# YOLOv8 PPE Safety Detection — Colab Training (STABLE)

**Status:** Production-ready
**Timeline:** ~4 hours
**GPU:** T4 or A100 recommended

## [1] Mount Google Drive & Setup

In [ ]:
from google.colab import drive
import os
import sys

print("[1.1] Mounting Google Drive...")
try:
    drive.mount('/content/drive')
    print("[1.1] PASS: Google Drive mounted")
except Exception as e:
    print(f"[1.1] FAIL: {e}")
    sys.exit(1)

print("[1.2] Creating workspace...")
os.makedirs('/content/workspace', exist_ok=True)
os.chdir('/content/workspace')
print(f"[1.2] PASS: Working directory = {os.getcwd()}")

print("\n[STEP 1] COMPLETE")

## [2] Install Dependencies

In [ ]:
import subprocess
import sys

print("[2.1] Installing core dependencies...")
deps = [
    'ultralytics==8.1.0',
    'opencv-python==4.8.1.78',
    'pyyaml==6.0',
    'albumentations==1.3.1',
    'roboflow>=1.1.0',
    'kaggle>=1.6.0',
    'huggingface-hub>=0.20.0',
    'inference-sdk>=0.9.0',   # Roboflow workflow inference
]

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q'] + deps,
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print(f"[2.1] WARNING: {result.stderr[:200]}")
else:
    print("[2.1] PASS: All dependencies installed")

print("\n[STEP 2] COMPLETE")

## [3] Clone Repository

In [ ]:
import subprocess
import os

print("[3.1] Cloning repository...")
result = subprocess.run(
    ['git', 'clone', '--depth=1', 
     'https://github.com/A-Kuo/Yolov8-Powered-Safety-Equipment-Detection-System.git',
     'project'],
    capture_output=True,
    text=True,
    cwd='/content/workspace'
)

if result.returncode != 0:
    print(f"[3.1] FAIL: {result.stderr}")
else:
    print("[3.1] PASS: Repository cloned")

print("[3.2] Checking files...")
if os.path.exists('/content/workspace/project/data/preprocessing'):
    print("[3.2] PASS: Preprocessing directory found")
else:
    print("[3.2] FAIL: Preprocessing directory not found")

print("\n[STEP 3] COMPLETE")

## [3B] Roboflow Workflow Backend — RECOMMENDED (No Training Required)

**Use this section instead of [9]-[10] if you have a Roboflow API key.**

The pre-built PPE Compliance Pipeline (`zGLpQAKajlvk32DknfR6`) runs two models server-side:
- **rfdetr-nano** — worker / person tracking
- **ppe-hgqzw/6** — fine-grained PPE detection (trained on 8,700+ real-world images)

One API call replaces the entire local two-model pipeline — no GPU training needed.

**Setup:** Set your API key in the cell below, then run [3B.1] → [3B.3].

In [ ]:
import os
import sys

# ── Configure your Roboflow API key ──────────────────────────────────────────
# Option A: set as env var before running (recommended)
#   export ROBOFLOW_API_KEY="rf-xxxx..."
# Option B: paste the key directly here (never commit to git)
ROBOFLOW_API_KEY = os.getenv("ROBOFLOW_API_KEY", "")

if not ROBOFLOW_API_KEY:
    print("[3B.1] WARNING: ROBOFLOW_API_KEY not set.")
    print("       Set it via: os.environ['ROBOFLOW_API_KEY'] = 'rf-...'")
else:
    os.environ["ROBOFLOW_API_KEY"] = ROBOFLOW_API_KEY
    print(f"[3B.1] PASS: API key configured ({ROBOFLOW_API_KEY[:8]}...)")

# Add project to sys.path so imports work
sys.path.insert(0, '/content/workspace/project')
print("[3B.1] PASS: Project path added")

In [ ]:
import numpy as np

print("[3B.2] Testing Roboflow workflow connection...")

try:
    from inference_sdk import InferenceHTTPClient

    client = InferenceHTTPClient(
        api_url="https://detect.roboflow.com",
        api_key=os.environ["ROBOFLOW_API_KEY"],
    )

    # Send a small blank frame to verify the connection
    dummy = np.zeros((64, 64, 3), dtype=np.uint8)
    from PIL import Image
    import io, base64
    buf = io.BytesIO()
    Image.fromarray(dummy).save(buf, format="JPEG")
    encoded = base64.b64encode(buf.getvalue()).decode()

    result = client.run_workflow(
        workspace_name="austins-workspace-gjnf8",
        workflow_id="zGLpQAKajlvk32DknfR6",
        images={"image": encoded},
    )

    print("[3B.2] PASS: Roboflow workflow reachable")
    print(f"       Response keys: {list(result[0].keys()) if result else 'empty'}")

except Exception as exc:
    print(f"[3B.2] FAIL: {exc}")
    print("       Check: ROBOFLOW_API_KEY is correct and inference-sdk is installed")

In [ ]:
import sys
import json

# ── Set VIDEO_PATH to your test video ────────────────────────────────────────
VIDEO_PATH = "/content/drive/MyDrive/test_video.mp4"   # ← change this
OUTPUT_DIR = "/content/workspace/roboflow_results"
MAX_FRAMES = 60   # ~2 seconds at 30 fps for a quick sanity check

print(f"[3B.3] Running Roboflow PPE pipeline on: {VIDEO_PATH}")
print(f"       Max frames: {MAX_FRAMES}  |  Output: {OUTPUT_DIR}")

try:
    from src.edge_deployment.roboflow_inference import RoboflowInference

    rf_client = RoboflowInference(
        api_key=os.environ["ROBOFLOW_API_KEY"],
        conf_threshold=0.25,
    )

    results = rf_client.process_video(
        video_path=VIDEO_PATH,
        max_frames=MAX_FRAMES,
        target_fps=5,   # sample at 5 fps to stay within API rate limits
    )

    print(f"\n[3B.3] PASS: {len(results)} frames processed")

    # Quick summary
    total_workers   = sum(len(r) for r in results)
    total_detections = sum(len(v) for r in results for v in r.values())
    print(f"       Workers detected (total): {total_workers}")
    print(f"       PPE items detected (total): {total_detections}")

    # Save results
    import os, json
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    serialisable = [
        {
            str(wid): [
                {"class": d.class_name, "conf": round(d.confidence, 3), "bbox": d.bbox}
                for d in items
            ]
            for wid, items in frame.items()
        }
        for frame in results
    ]
    with open(f"{OUTPUT_DIR}/detections.json", "w") as f:
        json.dump(serialisable, f, indent=2)

    metrics = rf_client.get_metrics()
    print(f"\n       Mean latency : {metrics.get('total_ms', {}).get('mean', 0):.0f} ms/frame")
    print(f"       Effective FPS: {metrics.get('fps', 0):.1f}")

    print(f"\n[3B.3] Results saved to {OUTPUT_DIR}/detections.json")
    print("[3B] ✓ Roboflow backend fully operational — skip [4]-[10] training steps")

except FileNotFoundError:
    print(f"[3B.3] INFO: Video not found at {VIDEO_PATH}")
    print("       Upload a video to Google Drive and update VIDEO_PATH above.")
except Exception as exc:
    print(f"[3B.3] FAIL: {exc}")
    raise

## [4] Setup Data Structure

In [ ]:
import shutil
import os

os.chdir('/content/workspace')

print("[4.1] Creating data directories...")
dirs = [
    'data/raw',
    'data/annotations',
    'data/processed/merged/train/images',
    'data/processed/merged/train/labels',
    'data/processed/merged/val/images',
    'data/processed/merged/val/labels',
    'data/processed/merged/test/images',
    'data/processed/merged/test/labels',
    'models/exports',
    'runs'
]

for d in dirs:
    os.makedirs(d, exist_ok=True)

print("[4.1] PASS: Directory structure created")

print("[4.2] Copying config files...")
try:
    shutil.copytree(
        '/content/workspace/project/data/annotations',
        '/content/workspace/data/annotations',
        dirs_exist_ok=True
    )
    shutil.copytree(
        '/content/workspace/project/config',
        '/content/workspace/config',
        dirs_exist_ok=True
    )
    print("[4.2] PASS: Configs copied")
except Exception as e:
    print(f"[4.2] FAIL: {e}")

print("\n[STEP 4] COMPLETE")

## [5] Download Ultralytics Construction-PPE

In [ ]:
from ultralytics import YOLO
import os
import glob

print("[5.1] Triggering Ultralytics auto-download...")
try:
    model = YOLO('yolov8n.pt')
    results = model.train(
        data='construction-ppe.yaml',
        epochs=0,
        exist_ok=True,
        verbose=False
    )
except Exception as e:
    print(f"[5.1] INFO: Expected error (epochs=0): {str(e)[:100]}")

print("[5.2] Finding dataset location...")
dataset_paths = glob.glob(
    os.path.expanduser('~/.cache/ultralytics/datasets/construction-ppe'),
    recursive=False
)

if dataset_paths:
    DATASET_PATH = dataset_paths[0]
    print(f"[5.2] PASS: Found at {DATASET_PATH}")
else:
    DATASET_PATH = os.path.expanduser('~/ultralytics/datasets/construction-ppe')
    print(f"[5.2] PASS: Using default path {DATASET_PATH}")

# Store for later
globals()['ULTRALYTICS_DATASET'] = DATASET_PATH
print(f"\n[STEP 5] COMPLETE")

## [6] Setup Dataset YAML

In [ ]:
import yaml
import os

print("[6.1] Creating merged dataset YAML...")

dataset_yaml = {
    'path': '/content/workspace/data/processed/merged',
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'nc': 10,
    'names': {
        0: 'worker',
        1: 'drone',
        2: 'safety_glasses',
        3: 'safety_goggles',
        4: 'hard_hat',
        5: 'regular_hat',
        6: 'hi_vis_vest',
        7: 'regular_clothing',
        8: 'work_boots',
        9: 'regular_shoes'
    }
}

try:
    with open('/content/workspace/dataset.yaml', 'w') as f:
        yaml.dump(dataset_yaml, f, default_flow_style=False)
    print("[6.1] PASS: dataset.yaml created")
except Exception as e:
    print(f"[6.1] FAIL: {e}")

print("\n[STEP 6] COMPLETE")

## [7] Populate with Ultralytics Data

In [ ]:
import shutil
import os
import glob

DATASET_PATH = globals().get('ULTRALYTICS_DATASET')

if not DATASET_PATH:
    print("[7.1] FAIL: No dataset path found")
else:
    print(f"[7.1] Copying Ultralytics data from {DATASET_PATH}...")
    
    try:
        # Copy train images
        src_train = os.path.join(DATASET_PATH, 'images', 'train')
        dst_train = '/content/workspace/data/processed/merged/train/images'
        
        if os.path.exists(src_train):
            imgs = glob.glob(os.path.join(src_train, '*'))
            for img in imgs[:100]:  # Limit for speed
                shutil.copy2(img, dst_train)
            print(f"[7.1] PASS: Copied {min(len(imgs), 100)} training images")
        
        # Copy train labels
        src_lbl = os.path.join(DATASET_PATH, 'labels', 'train')
        dst_lbl = '/content/workspace/data/processed/merged/train/labels'
        
        if os.path.exists(src_lbl):
            lbls = glob.glob(os.path.join(src_lbl, '*.txt'))
            for lbl in lbls[:100]:
                shutil.copy2(lbl, dst_lbl)
            print(f"[7.1] PASS: Copied {min(len(lbls), 100)} label files")
        
    except Exception as e:
        print(f"[7.1] WARNING: {e}")

print("\n[STEP 7] COMPLETE")

## [8] Download Pretrained Model

In [ ]:
from huggingface_hub import hf_hub_download
import os
import shutil

print("[8.1] Downloading pretrained PPE model...")

try:
    model_path = hf_hub_download(
        repo_id="keremberke/yolov8m-protective-equipment-detection",
        filename="best.pt",
    )
    print(f"[8.1] PASS: Downloaded to {model_path}")
    
    # Copy to models directory
    os.makedirs('/content/workspace/models', exist_ok=True)
    shutil.copy(model_path, '/content/workspace/models/pretrained_ppe_m.pt')
    print("[8.1] PASS: Copied to models/")
    
except Exception as e:
    print(f"[8.1] FAIL: {e}")

print("\n[STEP 8] COMPLETE")

## [9] Check GPU & Dataset

In [ ]:
from ultralytics import YOLO
import torch
import os

print("[9.1] Loading model...")

model_path = '/content/workspace/models/pretrained_ppe_m.pt'
fallback_path = 'yolov8m.pt'

try:
    if os.path.exists(model_path):
        model = YOLO(model_path)
        print(f"[9.1] PASS: Loaded pretrained PPE model")
    else:
        print(f"[9.1] INFO: Pretrained model not found, using yolov8m.pt")
        model = YOLO(fallback_path)
        print("[9.1] PASS: Loaded base yolov8m model")
except Exception as e:
    print(f"[9.1] FAIL: {e}")
    raise

print("\n[9.2] Starting fine-tuning training...")
print("Dataset: /content/workspace/dataset.yaml")
print("Epochs: 80 (Phase 2A: increased from 30 for better synthetic baseline)")
print("Batch size: 16")
print("LR: 0.001 (fine-tuning)")
print("Warmup: 5 epochs")
print("Freeze: 10 (transfer learning)")
print()

try:
    results = model.train(
        data='/content/workspace/dataset.yaml',
        epochs=80,  # Phase 2A: increased from 30 for better synthetic baseline
        imgsz=640,
        batch=16,
        device=0 if torch.cuda.is_available() else 'cpu',
        patience=10,
        save=True,
        project='/content/workspace/runs',
        name='ppe_v1',
        exist_ok=True,
        optimizer='SGD',
        lr0=0.001,
        lrf=0.01,
        momentum=0.937,
        weight_decay=0.0005,
        warmup_epochs=5,  # Phase 2A: increased from 3
        amp=True,
        freeze=10,
        hsv_h=0.010,
        hsv_s=0.50,
        hsv_v=0.40,
        degrees=15,
        translate=0.1,
        scale=0.50,
        fliplr=0.50,
        mosaic=1.0,
        mixup=0.1,
    )
    print("[9.2] PASS: Training complete")
    print(f"Results: /content/workspace/runs/ppe_v1")
except Exception as e:
    print(f"[9.2] FAIL: {e}")
    raise

print("\n[STEP 9] COMPLETE")

## [10] Start Fine-Tuning Training

In [ ]:
import subprocess
import shutil
import os

print("[5C] PHASE 2C: Merging Real Datasets with Class Remapping...")

# Copy merge script and config from project
try:
    os.makedirs('/content/workspace/data/preprocessing', exist_ok=True)
    
    shutil.copy(
        '/content/workspace/project/data/preprocessing/merge_real_datasets.py',
        '/content/workspace/data/preprocessing/merge_real_datasets.py'
    )
    print("[5C] Copied merge script")
    
    shutil.copy(
        '/content/workspace/project/data/annotations/class_mapping_public.yaml',
        '/content/workspace/data/annotations/class_mapping_public.yaml'
    )
    print("[5C] Copied class mapping config")
    
except Exception as e:
    print(f"[5C] WARNING: {e}")

# Check if real datasets were downloaded
roboflow_exists = os.path.exists('/content/workspace/data/raw/roboflow_construction')
kaggle_exists = os.path.exists('/content/workspace/data/raw/kaggle_sh17')

print(f"\n[5C] Roboflow available: {roboflow_exists}")
print(f"[5C] Kaggle available: {kaggle_exists}")

if roboflow_exists or kaggle_exists:
    print("\n[5C] Running merge script...")
    
    result = subprocess.run(
        ['/usr/bin/python3', '/content/workspace/data/preprocessing/merge_real_datasets.py',
         '--roboflow', '/content/workspace/data/raw/roboflow_construction',
         '--kaggle', '/content/workspace/data/raw/kaggle_sh17',
         '--output', '/content/workspace/data/processed/mixed'],
        capture_output=True,
        text=True
    )
    
    print(result.stdout)
    if result.returncode != 0:
        print(f"[5C] INFO: {result.stderr[:200]}")
    
    print("\n[5C] PASS: Datasets merged (if downloads succeeded)")
    print("[5C] Next: Use config/mixed_dataset.yaml for Phase 2C training")
    
else:
    print("\n[5C] INFO: Real datasets not downloaded, skipping merge")
    print("[5C]      Run Phase 2B cells above to download Roboflow + Kaggle")

print("\n[PHASE 2C] Complete - Datasets merged (optional)")

In [ ]:
import subprocess
import sys
import os

print("[5B.1] PHASE 2B: Downloading Roboflow Construction Safety (2,801 images)...")

try:
    from roboflow import Roboflow
    
    rf = Roboflow(api_key="roboflow_free")
    project = rf.workspace("roboflow-universe").project("construction-site-safety-yolov8")
    dataset = project.download("yolov8", location="/content/workspace/data/raw")
    
    print("[5B.1] PASS: Roboflow dataset downloaded")
    
except Exception as e:
    print(f"[5B.1] INFO: Roboflow download unavailable: {str(e)[:100]}")
    print("       → This is optional for Phase 2C. Continue without it if needed.")

print("\n[5B.2] PHASE 2B: Downloading Kaggle SH17 Dataset (8,099 images)...")
print("       Required: ~/.kaggle/kaggle.json (from https://www.kaggle.com/settings/account)")

try:
    import kaggle
    
    kaggle.api.dataset_download_files(
        'deeptech/sh17-dataset',
        path='/content/workspace/data/raw',
        unzip=True
    )
    
    print("[5B.2] PASS: Kaggle SH17 dataset downloaded")
    
except Exception as e:
    print(f"[5B.2] INFO: Kaggle download unavailable: {str(e)[:100]}")
    print("       → Setup: https://www.kaggle.com/settings/account")
    print("       → Create API Token, save to ~/.kaggle/kaggle.json")
    print("       → This is optional for Phase 2C. Continue without it if needed.")

print("\n[PHASE 2B] Complete - Real datasets downloaded (optional)")

## [OPTIONAL] Phase 2B/2C: Real Data Collection & Fine-Tuning

**Phase 2B:** Download real datasets (Roboflow + Kaggle)  
**Phase 2C:** Merge datasets and train on mixed real+synthetic data

To use Phase 2B/2C, run the cells below. Otherwise, skip to [10].

In [ ]:
from ultralytics import YOLO
import os

print("[10.1] Loading pretrained model...")

try:
    model = YOLO('/content/workspace/models/pretrained_ppe_m.pt')
    print("[10.1] PASS: Model loaded")
except Exception as e:
    print(f"[10.1] FAIL: {e}")
    raise

print("\n[10.2] Starting fine-tuning...")
print("Dataset: /content/workspace/dataset.yaml")
print("Epochs: 30")
print("Batch size: 16")
print("LR: 0.001 (fine-tuning)")
print("Freeze: 10 (transfer learning)")
print()

try:
    results = model.train(
        data='/content/workspace/dataset.yaml',
        epochs=30,
        imgsz=640,
        batch=16,
        device=0 if torch.cuda.is_available() else 'cpu',
        patience=10,
        save=True,
        project='/content/workspace/runs',
        name='ppe_v1',
        exist_ok=True,
        optimizer='SGD',
        lr0=0.001,
        lrf=0.01,
        momentum=0.937,
        weight_decay=0.0005,
        warmup_epochs=3,
        amp=True,
        freeze=10,
        hsv_h=0.010,
        hsv_s=0.50,
        hsv_v=0.40,
        degrees=15,
        translate=0.1,
        scale=0.50,
        fliplr=0.50,
        mosaic=1.0,
        mixup=0.1,
    )
    print("[10.2] PASS: Training complete")
    print(f"Results: /content/workspace/runs/ppe_v1")
except Exception as e:
    print(f"[10.2] FAIL: {e}")
    raise

print("\n[STEP 10] COMPLETE")

## [11] Export Models

In [ ]:
from ultralytics import YOLO
import shutil
import os

print("[11.1] Loading best model...")

try:
    best_model = YOLO('/content/workspace/runs/ppe_v1/weights/best.pt')
    print("[11.1] PASS: Best model loaded")
except Exception as e:
    print(f"[11.1] FAIL: {e}")
    raise

print("\n[11.2] Exporting to ONNX...")

try:
    onnx_path = best_model.export(format='onnx', imgsz=640, opset=12)
    print(f"[11.2] PASS: {onnx_path}")
    shutil.copy(onnx_path, '/content/workspace/models/exports/ppe_detector.onnx')
except Exception as e:
    print(f"[11.2] WARNING: {e}")

print("\n[11.3] Copying PyTorch model...")

try:
    shutil.copy(
        '/content/workspace/runs/ppe_v1/weights/best.pt',
        '/content/workspace/models/exports/ppe_detector.pt'
    )
    print("[11.3] PASS: PyTorch model copied")
except Exception as e:
    print(f"[11.3] FAIL: {e}")

print("\n[STEP 11] COMPLETE")

## [12] Save to Google Drive

In [ ]:
import shutil
import os

print("[12.1] Saving to Google Drive...")

drive_path = '/content/drive/MyDrive/YOLOv8_PPE_Training'
os.makedirs(drive_path, exist_ok=True)

try:
    # Copy models
    shutil.copytree(
        '/content/workspace/models/exports',
        f'{drive_path}/models',
        dirs_exist_ok=True
    )
    print(f"[12.1] PASS: Models saved")
except Exception as e:
    print(f"[12.1] WARNING: {e}")

print(f"\nLocation: {drive_path}")
print("\n[STEP 12] COMPLETE")

## [13] Final Report

In [ ]:
import os
import glob

print("="*60)
print("TRAINING PIPELINE COMPLETE")
print("="*60)

print("\n[OUTPUTS]")
print(f"PyTorch model: /content/workspace/models/exports/ppe_detector.pt")
print(f"ONNX model: /content/workspace/models/exports/ppe_detector.onnx")
print(f"Training logs: /content/workspace/runs/ppe_v1/")

print("\n[AVAILABLE MODELS]")
for f in glob.glob('/content/workspace/models/exports/*'):
    size_mb = os.path.getsize(f) / (1024*1024)
    print(f"  {os.path.basename(f)} ({size_mb:.1f} MB)")

print("\n[NEXT STEPS]")
print("1. Download models from: /content/drive/MyDrive/YOLOv8_PPE_Training/")
print("2. Test on local warehouse video")
print("3. Deploy to Snapdragon device")
print("\n" + "="*60)